This project aims to identify areas in New York City with a high density of ride-hailing (VTC) requests during the years 2014 and 2015.

In [ ]:
import zipfile 
import pandas as pd
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from sklearn.cluster import DBSCAN
import plotly.graph_objects as go
import urllib.request
import zipfile
import geopandas as gpd
from sktime.clustering.spatio_temporal import STDBSCAN
import plotly.express as px
import holidays
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import joblib

# 1. Data extraction

First, we will extract the different datasets. Each of them corresponds to a specific period in 2014 or 2015.

In [ ]:
# The datasets are in zip files.
uber_14 = 'uber-trip-data.zip'

with zipfile.ZipFile(uber_14, 'r') as zip_ref:
    zip_ref.extractall('.')

# The file for 2015 is also a zip file.
uber_15 = r'C:\Users\axelv\Documents\Jedha\DATA_SCIENCE\FULLSTACK\Projet\05_Uns_Mach_Learn\uber\uber-trip-data\uber-raw-data-janjune-15.csv.zip'

with zipfile.ZipFile(uber_15, 'r') as zip_ref:
    zip_ref.extractall('.')

In [ ]:
# We import the unzipped datasets.
path = r'C:\Users\axelv\Documents\Jedha\DATA_SCIENCE\FULLSTACK\Projet\05_Uns_Mach_Learn\uber\uber-trip-data'
df_lookup = pd.read_csv(f'{path}/taxi-zone-lookup.csv')
df_apr = pd.read_csv(f'{path}/uber-raw-data-apr14.csv')
df_aug = pd.read_csv(f'{path}/uber-raw-data-aug14.csv')
df_jul = pd.read_csv(f'{path}/uber-raw-data-jul14.csv')
df_jun = pd.read_csv(f'{path}/uber-raw-data-jun14.csv')
df_may = pd.read_csv(f'{path}/uber-raw-data-may14.csv')
df_sep = pd.read_csv(f'{path}/uber-raw-data-sep14.csv')
df_janjune_15 = pd.read_csv(f'{path}/uber-raw-data-janjune-15.csv')

# 2. Dataset analysis 

## 2.1. Overview 

Before focusing on a specific month, let’s take a look at the "Taxi Zone Lookup" dataset.

In [ ]:
display(df_lookup.head(50))

In [ ]:
display(df_lookup.info())

The lookup dataset provides the boroughs and zones frequented by users of the application.

Let’s take a look at the dataset for the month of April.

In [ ]:
display(df_apr.head(100))

In [ ]:
display(df_apr.info(20))

In [ ]:
df_apr.isna().sum() * 100

The dataset for the month of April records each trip made during that month. The boroughs and zones are not directly referenced, so we will need to link them using the provided latitude and longitude, which correspond to the point where the passenger ask for a trip. 

Before doing that, we will first isolate one hour from a single day to simplify the task at the beginning.

# 3. Work on a single jour
## 3.1. Selection of the day and hour

In [ ]:
# First, we will convert the time column into datetime format.
df_apr['Date/Time'] = pd.to_datetime(df_apr['Date/Time'])

# Let’s use noon as our working hour
df_apr_01_12 = df_apr[(df_apr['Date/Time'].dt.hour == 12) & (df_apr['Date/Time'].dt.day == 1)]
display(df_apr_01_12.head())

## 3.2. DBSCAN

We want to identify the areas with the highest density of ride requests. This is therefore a dimensionality reduction problem. The variables involve geospatial data, so DBSCAN is the most suitable algorithm for this task.

## 3.2.1 Data preparation

Normally, DBSCAN requires data to be normalized before use. However, here we are dealing with spatial data that we want to visualize directly on a map. For that reason, we will not normalize the data.


First, we will clean the data. DBSCAN might incorrectly interpret minutes as spatial proximity, so we will remove them. Then, we will select each hour separately before feeding the data into DBSCAN.

We will also remove the “base” column, as it does not provide any useful information for DBSCAN and could distort its interpretation.

In [ ]:
# Selecting the relevant information.
X_apr_01_12 = df_apr_01_12[['Lat', 'Lon']]

## 3.2.2. Searching for epsilon

Working with DBSCAN mainly comes down to finding the right values for epsilon and min_samples.
Let’s start by determining epsilon. It corresponds to the radius of the neighborhood used to define clusters.

In [ ]:
'''nb = NearestNeighbors(n_neighbors=2)  # 2 because we are looking for the nearest neighbor
nb.fit(X_apr_01_12)
distance, _ = nb.kneighbors(X_apr_01_12)    # It returns two things: the distance and the identifier
plt.plot(np.sort(distance[:,1]))'''

We can observe an elbow around 0.01. One degree of latitude or longitude corresponds to approximately 100 km (this varies between the equator and the poles). Our epsilon therefore represents a radius of 0.01, i.e. roughly 1 km, which seems coherent.

## 3.2.3 Searching for min_samples

Now let’s determine min_samples. This represents the density threshold—in this case, the number of Uber requests per zone.

There are several empirical rules for choosing min_samples. If n is the number of dimensions in our dataset:

- At minimum, it should be n + 1
- A common recommendation is 2 × n
- Or ln(n) (Too small here)
- Or simply 5, which often works well and is also the default value

Let’s try these values using the silhouette score.

In [ ]:
'''for min_samples in[3, 4, 5,10,20,30,40,50,60,70,80,90,100]: # Values to test
    db = DBSCAN(eps=0.01, min_samples=min_samples)
    labels = db.fit_predict(X_apr_01_12)    # Extraction of labels
    if len(np.unique(labels)) > 1 :     # Because silhouette_score does not accept a clustering model directly
        print(min_samples, silhouette_score(X_apr_01_12, labels))   # List of clusters found
    '''

We see good results around 10. Let’s explore values around that number.

In [ ]:
'''for min_samples in[6,7,8,9,10,11,12,13,14,15,16,17,18]:
    db = DBSCAN(eps=0.01, min_samples=min_samples)
    labels = db.fit_predict(X_apr_01_12)
    if len(np.unique(labels)) > 1 :
        print(min_samples, silhouette_score(X_apr_01_12, labels))'''

Great, we choose an epsilon of 0.01 and min_samples of 14.

In [ ]:
'''# Instantiation of DBSCAN with our parameters
db = DBSCAN(eps=0.01, min_samples=14, metric="euclidean", algorithm="auto")   # Euclidean distance is the most suitable for geospatial data.  
db.fit(X_apr_01_12)

# Instantiation of the schema
fig = go.Figure()
for i in np.unique(db.labels_):
    label = X_apr_01_12[db.labels_ == i].values     # .values because X_apr_01_12 is a dataFrame
    fig.add_trace(go.Scatter(x = label[:,0], y = label[:,1], mode="markers", name="Cluster {}".format(i)))
fig.show()'''

Let’s look on a map to see what this cluster corresponds to.

In [ ]:
'''import plotly.express as px

df_plot = X_apr_01_12.copy()
df_plot['cluster'] = db.labels_.astype(str)

fig = px.scatter_mapbox(df_plot, lat='Lat', lon='Lon', color='cluster',
                        mapbox_style='open-street-map', zoom=11)
fig.show()'''

This corresponds to the island of Manhattan. It is well delimited; the cluster did not include any points outside the island, which seems consistent.

## 3.2. Link with geographic zones


Now the goal is to transform this cluster into a directly identifiable geographic zone.

The first dataset, Taxi Zone Lookup, appears to be the official shapefile of TLC zones in New York.

We will therefore use GeoPandas to retrieve the LocationID of the clusters. Finally, we will merge it with the Taxi Zone Lookup dataset.

In [ ]:
path_taxi_zone = r'C:\Users\axelv\Documents\Jedha\DATA_SCIENCE\FULLSTACK\Projet\05_Uns_Mach_Learn\uber\taxi_zones'
taxi_zone = gpd.read_file(path_taxi_zone + r'\taxi_zones.shp')

In [ ]:
display(taxi_zone.head())

Perfect, the file indeed contains the zone names associated with their geometry. We can now link it with our dataset.

In [ ]:
display(df_lookup.head(50))

It’s the same dataset, but with added location information for the new dataset of taxi zones. We will therefore use the taxi_zone dataset from now on.

The GPS coordinates we have are in degrees. The NYC shapefile is in feet. We will therefore convert feet into degrees.

In [ ]:
taxi_zone = taxi_zone.to_crs('EPSG:4326')

This will give us a geometry column, which is necessary to perform the join.

In [ ]:
X_apr_01_12_zones = gpd.GeoDataFrame(X_apr_01_12, 
    geometry=gpd.points_from_xy(X_apr_01_12['Lon'], X_apr_01_12['Lat']),
    crs='EPSG:4326')

In [ ]:
# For each customer, assign their corresponding zone.
df_apr_01_12_zones = gpd.sjoin(X_apr_01_12_zones, taxi_zone[['zone', 'borough', 'LocationID', 'geometry']], how='left', predicate='within')
display(df_apr_01_12_zones.head())

Alright, now that everything is set, we can, for example, look at which zones have the highest number of trips.

In [ ]:
df_apr_01_12_zones['zone'].value_counts().head(10)

These are indeed zones located within the cluster that DBSCAN had previously identified.

## 3.3. Link with the temporal data

### 2.3.1. ST-DBSCAN

Now that we’ve found a way to identify areas with high demand, we need to link them to periods of high demand.      

However, spatial and temporal data are not directly comparable to each other. We therefore need to find a way to cluster data using two completely independent conditions.      

That’s why we’ll use ST-DBSCAN, which is designed for this kind of problem. It allows us to define a second epsilon, introducing an additional independent condition for forming clusters. This makes it particularly well-suited for handling spatio-temporal data. [ST-DBSCAN documentation](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.clustering.spatio_temporal.STDBSCAN.html)

#### 2.3.1.1. Specificities of ST-DBSCAN.

The documentation tells us that ST-DBSCAN expects a very specific time format. It allows us to link each ID to its associated timestamp.        

We need to convert time into minutes and create a MultiIndex. This index should include both the one related to each trip and the one related to minutes. Without this, ST-DBSCAN would not be able to cluster the temporal data.

In [ ]:
# We will also include American events
# This will help us better understand transportation flow patterns.
us_holidays = holidays.US(years=[2014, 2015])

# Create minutes column
df_apr['minutes'] = df_apr['Date/Time'].dt.hour * 60 + df_apr['Date/Time'].dt.minute
df_apr['month'] = 'avril' # to enable the processing of all months afterwards
# Copy of latitude et longitude
df_apr['date'] = df_apr['Date/Time'].dt.date  # We keep the date
df_apr['event'] = df_apr['Date/Time'].dt.date.map(us_holidays)  # We add event. This will be useful if we want to find patterns
df_apr_gps = df_apr[['Lat', 'Lon']].copy()

# Replace the simple index with a MultiIndex.
df_apr_gps.index = pd.MultiIndex.from_arrays([df_apr.index, df_apr['minutes']], names=['object_id', 'time'])

We set eps2 = 30.           
This corresponds to a 30-minute time window. It allows us to capture a certain level of precision in how order density evolves over time.

In [ ]:
'''# Initialization of ST-DBSCAN
st_db = STDBSCAN(eps1=0.01, eps2=30, min_samples=14, metric="euclidean") 
st_db.fit(df_apr_gps)
labels = st_dbscan.labels_'''

#### 2.3.1.1. Conclusion with ST-DBSCAN 
We reach an interesting conclusion: the number of points is too large for ST-DBSCAN, which tries to compute a full distance matrix over all these points.

As a result, DBSCAN turns out to be more suitable for large-scale datasets. We therefore go back to using it instead.

## 4. Filtering zones by time slots in one month

We will apply the following approach: we will work with 30-minute time intervals. This gives us a fairly precise view of how density evolves over time.

We will then compute the corresponding dense zones, and repeat this process for each month.

### 4.1. Function definition

We will therefore create a function for a 30-minute time window. We will then apply it 48 times over a full day.

In [ ]:
def get_hot_zones(df, start_min, end_min, taxi_zone):
    # Filter by minutes
    mask = (df['minutes'] >= start_min) & (df['minutes'] < end_min)
    df_slice = df[mask].copy()
    
    # If there are too few points, DBSCAN becomes unstable
    # We therefore enforce a minimum density threshold
    if len(df_slice) < 14:
        return None
    
    # DBSCAN
    X = df_slice[['Lat', 'Lon']]
    db = DBSCAN(eps=0.01, min_samples=14, metric="euclidean")
    df_slice['cluster'] = db.fit_predict(X)
    
    # Convert to GeoDataFrame
    # We transform the points into geographic objects
    gdf = gpd.GeoDataFrame(df_slice,
        geometry=gpd.points_from_xy(df_slice['Lon'], df_slice['Lat']),
        crs='EPSG:4326')
    # Spatial join with taxi zones
    result = gpd.sjoin(gdf, taxi_zone[['zone', 'borough', 'LocationID', 'geometry']],
                       how='left', predicate='within')
    
    # We keep track of the time window
    # Otherwise we would lose the temporal evolution
    result['start_min'] = start_min
    result['end_min'] = end_min
    # We produce an exploitable dataset
    return result[['Lat', 'Lon', 'cluster', 'zone', 'borough', 'start_min', 'end_min', 'month', 'date', 'event']]


Now that the function is defined, we need to loop over it for each day of the month.

In [ ]:
period = range(0, 1440, 30)   # 1440 = number of minutes in one day
resultats = []

# For each 30-minute period, we apply get_hot_zones
for start in period:
    hot_zones = get_hot_zones(df_apr, start, start+30, taxi_zone)
    # We skip empty periods
    if hot_zones is not None:
        resultats.append(hot_zones)

# We stack the results
df_final = pd.concat(resultats, ignore_index=True)

In [ ]:
# Building the aggregated dataset
df_hot_time_apr = (df_final[df_final['cluster'] != -1]  # We keep only meaningul clusters
                # We group geaographic zone and time windows of 30 minutes 
               .groupby(['zone', 'start_min'])
               # We count the number of trip in the zone 
               .agg(count=('Lat', 'count'), Lat=('Lat', 'mean'), Lon=('Lon', 'mean'))
               # Convert the grouped result in a table
               .reset_index())

# Dynamic visualization
fig = px.scatter_map(df_hot_time_apr, lat='Lat', lon='Lon', size='count',
                     hover_name='zone', animation_frame='start_min', zoom=10)
fig.show()



Here is the dynamic map of the zones with the highest demand for the month of April.

# 5. Application to all months.

## 5.1 Preparation of all months.

We can therefore apply this loop across all months.

In [ ]:

# Creat dictionary with all monthly dataset
dfs = {
    'avril': df_apr, 'mai': df_may, 'juin': df_jun,
    'juillet': df_jul, 'août': df_aug, 'septembre': df_sep
}

# Preprocessing loop
for month, df in dfs.items():
    # DateTime conversion
    df['Date/Time'] = pd.to_datetime(df['Date/Time'])
    # Convert tile into minutes
    df['minutes'] = df['Date/Time'].dt.hour * 60 + df['Date/Time'].dt.minute
    # Add month label
    df['month'] = month
    # We keep the dates
    df['date'] = df['Date/Time'].dt.date
    # We add the events. 
    # This will be useful later if we want to find patterns in these flows.
    df['event'] = df['Date/Time'].dt.date.map(us_holidays)

# Final concatenation
df_all = pd.concat(dfs.values(), ignore_index=True)

In [ ]:
resultats = []

# Loop over each month
for month, df_month in df_all.groupby('month'):
    # Loop over time windows
    for start in range(0, 1440, 30):
        # Apply hotspot detection
        hot_zones = get_hot_zones(df_month, start, start+30, taxi_zone)
        # Ignore empty results
        if hot_zones is not None:
            resultats.append(hot_zones)

# Final merge
df_final_all = pd.concat(resultats, ignore_index=True)

In [ ]:
import plotly.express as px

# We build the aggregated dataset
df_final_all['event'] = df_final_all['event'].fillna('None')
df_hot_time = (df_final_all[df_final_all['cluster'] != -1]
               .groupby(['zone', 'month', 'start_min', 'date', 'event'])
               .agg(count=('Lat', 'count'), Lat=('Lat', 'mean'), Lon=('Lon', 'mean'))
               .reset_index())

fig = px.scatter_map(df_hot_time, lat='Lat', lon='Lon', size='count',
                     hover_name='zone', animation_frame='start_min',
                     color='month', zoom=10)
fig.show()

Here is the map of all combined zones. We can also generate maps for each individual zone for a more detailed view.

In [ ]:
# Map for each month
for month in df_hot_time['month'].unique():
    df_month = df_hot_time[df_hot_time['month'] == month]
    fig = px.scatter_map(df_month, lat='Lat', lon='Lon', size='count',
                         hover_name='zone', animation_frame='start_min',
                         zoom=10, title=f'Hot zones - {month}')
    fig.show()


Now let’s focus on the jan-june 2015 dataset, which has a different structure.

In [ ]:
#Let’s take a closer look.
display(df_janjune_15.head())
display(df_janjune_15.info())

There are no latitude and longitude values here, only LocationIDs. Using these IDs, we can directly link the data to the corresponding taxi zones.

In [ ]:
# Zone centroids
taxi_zone['Lat'] = taxi_zone.geometry.centroid.y
taxi_zone['Lon'] = taxi_zone.geometry.centroid.x

In [ ]:
# Preparation
df_janjune_15['Pickup_date'] = pd.to_datetime(df_janjune_15['Pickup_date'])
df_janjune_15['minutes'] = df_janjune_15['Pickup_date'].dt.hour * 60 + df_janjune_15['Pickup_date'].dt.minute
df_janjune_15['month'] = df_janjune_15['Pickup_date'].dt.strftime('%B %Y')
df_janjune_15['date'] = df_janjune_15['Pickup_date'].dt.date
df_janjune_15['event'] = df_janjune_15['date'].map(us_holidays) # For the futur machine learning

In [ ]:
# Joining and aggregation
df_janjune_15['event'] = df_janjune_15['event'].fillna('None')
df_hot_15 = (df_janjune_15
             .merge(taxi_zone[['LocationID', 'zone', 'Lat', 'Lon']], left_on='locationID', right_on='LocationID')
             .groupby(['zone', 'month', 'minutes', 'event'])
             .agg(count=('locationID', 'count'), Lat=('Lat', 'mean'), Lon=('Lon', 'mean'))
             .reset_index()
             .rename(columns={'minutes': 'start_min'}))

In [ ]:
for month in df_hot_15['month'].unique():
    df_month = df_hot_15[df_hot_15['month'] == month]
    fig = px.scatter_map(df_month, lat='Lat', lon='Lon', size='count',
                         hover_name='zone', animation_frame='start_min',
                         zoom=10, title=f'Hot zones - {month}')
    fig.show()

Good, now we can combine the 2014 and 2015 datasets into a single clean table.

In [ ]:
df_hot_time['source'] = '2014'
df_hot_15['source'] = '2015'

df_all = pd.concat([df_hot_time, df_hot_15], ignore_index=True)
display(df_all.head())
display(df_all.info())

# 6. Machine learning model

Now that we have this data in a clean table, we can train a model to predict hot zones.

We are facing two issues: if we want a reliable model, we are missing key events that influence the decision to take a ride, such as **weather conditions**.    
We made sure to add American events, which are necessary to understand human mobility flows.

# 6.1. Preparation

We want a binary classification (hot zone or not). We will therefore use XGBoost, which performs well on this type of tabular data.

In [ ]:
# We will now look at how to set the threshold level for a hot zone
df_all['count'].describe()

The median is 1.5. Each zone therefore receives about 1 to 2 requests every 30 minutes. We will define a hot zone as roughly three times this value, i.e. around 5 ride requests.

In [ ]:
df_all['count'].hist(bins=50)

In [ ]:
df_all['is_hot'] = (df_all['count'] >= 5).astype(int)
df_all['is_hot'].value_counts()

Good — we have only 3% hot zones, so this is an imbalanced dataset. This will need to be taken into account when training the model.

In [ ]:
X = df_all[['zone', 'start_min', 'event']]
y = df_all['is_hot']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ['zone', 'event'])
], remainder='passthrough')

# 6.2. Training

We will train the model while taking the dataset imbalance into account.

In [ ]:
neg = df_all['is_hot'].value_counts()[0]
pos = df_all['is_hot'].value_counts()[1]

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(scale_pos_weight=neg/pos, random_state=42))
])

pipeline.fit(X_train, y_train)

In [ ]:
y_pred = (pipeline.predict_proba(X_test)[:, 1] >= 0.6).astype(int)
print(classification_report(y_test, y_pred))

In [ ]:
# Model exportation
joblib.dump(pipeline, 'model.pkl')

Overall, the model shows strong metrics, with a precision of 84% and a recall of 79%.

However, the dataset is missing essential information. Weather conditions and events specific to New York City significantly influence transport flows, yet they are currently absent. If Uber wants to build a more effective model, they should also collect this type of data. Absent these data points, the model’s deployment provides little practical value.

# 7. Clean dataset for Streamlit.

We will map the minute counter back to real timestamps so that we can present these maps properly.

In [ ]:
df_all['heure'] = df_all['start_min'].apply(lambda m: f"{m//60:02d}:{m%60:02d}")

In [ ]:
df_all.to_csv('hot_zones.csv', index=False)

and we are exporting the dataset to create the Streamlit dashboard, available at this URL: [Dashboard](https://huggingface.co/spaces/Alvlt/uber-hot-zones)
